# Week 88, Day 6: Sprint review

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w088d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('waymark_domain.py', 'w', encoding='utf-8') as f:
    f.write('import heapq\n\n\nclass UnknownPlace(LookupError):\n    pass\n\n\nclass NoRoute(Exception):\n    pass\n\n\ndef naismith_minutes(km, climb_m):\n    return round(km / 5 * 60 + climb_m / 600 * 60)\n\n\nclass Route:\n    def __init__(self, stops, km, climb_m):\n        self.stops, self.km, self.climb_m = list(stops), km, climb_m\n\n    @property\n    def minutes(self):\n        return naismith_minutes(self.km, self.climb_m)\n\n    def __repr__(self):\n        return f"Route({self.stops}, {self.km} km, {self.climb_m} m)"\n\n\nclass PathNetwork:\n    def __init__(self, heights):\n        self.heights = dict(heights)\n        self.paths = {name: {} for name in self.heights}\n\n    def _check(self, *names):\n        for name in names:\n            if name not in self.heights:\n                raise UnknownPlace(name)\n\n    def add_path(self, a, b, km):\n        self._check(a, b)\n        if km <= 0:\n            raise ValueError("km must be positive")\n        self.paths[a][b] = km\n        self.paths[b][a] = km\n\n    def climb(self, stops):\n        return sum(max(0, self.heights[b] - self.heights[a]) for a, b in zip(stops, stops[1:]))\n\n    def shortest(self, start, end):\n        self._check(start, end)\n        dist, prev, done = {start: 0}, {}, set()\n        heap = [(0, start)]\n        while heap:\n            d, node = heapq.heappop(heap)\n            if node in done:\n                continue\n            done.add(node)\n            if node == end:\n                break\n            for nxt, km in sorted(self.paths[node].items()):\n                if d + km < dist.get(nxt, float("inf")):\n                    dist[nxt] = d + km\n                    prev[nxt] = node\n                    heapq.heappush(heap, (d + km, nxt))\n        if end not in done:\n            raise NoRoute(f"no route from {start} to {end}")\n        stops = [end]\n        while stops[-1] != start:\n            stops.append(prev[stops[-1]])\n        stops.reverse()\n        return Route(stops, round(dist[end], 2), self.climb(stops))\n')
with open('waymark_map.py', 'w', encoding='utf-8') as f:
    f.write('from waymark_domain import PathNetwork\n\nHEIGHTS = {"Mill": 40, "Weir": 55, "Church": 80, "Ridge": 310, "Farm": 120, "Wood": 150, "Lake": 60, "Island": 5}\nPATHS = [("Mill", "Weir", 1.5), ("Weir", "Church", 2.0), ("Mill", "Church", 4.0), ("Church", "Ridge", 3.0),\n         ("Weir", "Farm", 2.5), ("Farm", "Ridge", 2.0), ("Farm", "Wood", 1.5), ("Wood", "Lake", 2.0), ("Lake", "Mill", 3.5)]\n\n\ndef sample_network():\n    network = PathNetwork(HEIGHTS)\n    for a, b, km in PATHS:\n        network.add_path(a, b, km)\n    return network\n')
with open('cli.py', 'w', encoding='utf-8') as f:
    f.write('import argparse\nimport contextlib\nimport io\nfrom waymark_domain import NoRoute, UnknownPlace\nfrom waymark_map import sample_network\n\n\ndef main(argv, out):\n    parser = argparse.ArgumentParser(prog="waymark")\n    sub = parser.add_subparsers(dest="command", required=True)\n    p = sub.add_parser("plan")\n    p.add_argument("start")\n    p.add_argument("end")\n    try:\n        with contextlib.redirect_stderr(io.StringIO()):\n            args = parser.parse_args(argv)\n    except SystemExit:\n        out.write("usage: waymark plan START END\\n")\n        return 2\n    try:\n        route = sample_network().shortest(args.start, args.end)\n    except UnknownPlace as e:\n        out.write(f"unknown place: {e.args[0]}\\n")\n        return 2\n    except NoRoute as e:\n        out.write(f"{e}\\n")\n        return 1\n    hours, minutes = divmod(route.minutes, 60)\n    out.write(f"{\' -> \'.join(route.stops)}: {round(route.km, 1)} km, {hours} h {minutes:02d} min\\n")\n    return 0\n')
with open('demo.txt', 'w', encoding='utf-8') as f:
    f.write('$ waymark plan Mill Ridge\nMill -> Weir -> Farm -> Ridge: 6.0 km, 1 h 39 min\n$ waymark plan Ridge Mill\nRidge -> Farm -> Weir -> Mill: 6.0 km, 1 h 12 min\n$ waymark plan Mill Moon\nunknown place: Moon\n[exit 2]\n$ waymark plan Mill Island\nno route from Mill to Island\n[exit 1]\n')

**Predict before you run:** what will this print?



In [ ]:
planned = {"S3": 3, "S5": 5, "S6": 2, "S8": 1}
done_on = {"S3": 2, "S6": 3, "S8": 5}
print(sum(planned[s] for s in done_on), [s for s in planned if s not in done_on])

**Predict before you run:** what will this print?



In [ ]:
planned = {"S3": 3, "S5": 5, "S6": 2, "S8": 1}
done_on = {"S3": 2, "S6": 3, "S8": 5}
total = sum(planned.values())
print([total - sum(planned[s] for s, d in done_on.items() if d <= day) for day in range(1, 6)])

**Predict before you run:** what will this print?



In [ ]:
import math
history = [8, 6]
average = sum(history) / len(history)
print(average, math.ceil(20 / average), math.ceil(20 / min(history)), math.ceil(20 / max(history)))

**Predict before you run:** what will this print?



In [ ]:
import io
from cli import main
out = io.StringIO()
code = main(["plan", "Weir", "Lake"], out)
print(code, out.getvalue().strip())

**Predict before you run:** what will this print?



In [ ]:
lines = open("demo.txt").read().splitlines()
print(sum(line.startswith("$ ") for line in lines), [line for line in lines if line.startswith("[exit")])

### Exercise: What carries over

Write `sprint_outcome(planned, done_on)`: `planned` maps story ids to points (in plan order) and `done_on` maps the stories that were finished to the day they were finished. Return `{"done": [...], "carried": [...], "points_done": N, "points_carried": M}` with ids in plan order.

In [ ]:
def sprint_outcome(planned, done_on):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_outcome():\n    \"done and carried\"\n    planned = {\"S3\": 3, \"S5\": 5, \"S6\": 2, \"S8\": 1}\n    assert sprint_outcome(planned, {\"S8\": 5, \"S3\": 2, \"S6\": 3}) == {\"done\": [\"S3\", \"S6\", \"S8\"], \"carried\": [\"S5\"], \"points_done\": 6, \"points_carried\": 5}\n    assert sprint_outcome(planned, {}) == {\"done\": [], \"carried\": [\"S3\", \"S5\", \"S6\", \"S8\"], \"points_done\": 0, \"points_carried\": 11}")

### Exercise: The burndown from finish days

Write `remaining_by_day(planned, done_on, days)`: a list of `days + 1` numbers: the points not yet finished at the start (day 0) and at the end of each day from 1 to `days`. A story counts as finished at the end of the day in `done_on`.

In [ ]:
def remaining_by_day(planned, done_on, days):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_burn():\n    \"points left each evening\"\n    planned = {\"S3\": 3, \"S5\": 5, \"S6\": 2, \"S8\": 1}\n    assert remaining_by_day(planned, {\"S3\": 2, \"S6\": 3, \"S8\": 5}, 5) == [11, 11, 8, 6, 6, 5]\n    assert remaining_by_day(planned, {}, 2) == [11, 11, 11]")

### Exercise: Sprint report

Write `sprint_report(name, planned, done_on, days, history, backlog_points, version, messages, date)` using the given helpers. `planned` maps story ids to points; `done_on` gives finishing days; `history` is the velocity of earlier sprints; `backlog_points` counts the points still in the backlog **excluding** carried-over stories; `version` is the current release; `messages` are the sprint's commit messages. Return a dict with `velocity` (points done), `history` (the old history plus this sprint), `carried`, `remaining` (the burndown list), `forecast` (from `forecast` with the new history, for the backlog **plus** the carried points), `release` (the new version, or `None` if nothing to release) and `text`, the review written exactly like this (lines in this order):

```
# NAME review
Done: S3, S6, S8 (6 points)
Carried over: S5 (5 points)
Burndown: 11 11 8 6 6 5
Velocity: 6 (average of last 3: 7.0)
Forecast for 25 points: 4 sprints (best 4, worst 5)
Release: 0.2.0
```

where `Done:` and `Carried over:` say `none (0 points)` when empty, the worst case is `unknown` when the slowest recent sprint finished nothing (the given `forecast` returns `None` for it), the average comes from `velocity(new_history)`, and the release line says `Release: nothing to release` when there's none.

In [ ]:
import math
import re

def velocity(history, window=3):
    recent = history[-window:]
    if not recent:
        return 0.0, 0, 0
    return round(sum(recent) / len(recent), 1), min(recent), max(recent)

def forecast(points, history, window=3):
    average, low, high = velocity(history, window)
    if average == 0:
        raise ValueError("no velocity yet")
    return {"expected": math.ceil(points / average), "best": math.ceil(points / high),
            "worst": math.ceil(points / low) if low else None}

HEADER = re.compile(r"(?P<type>[a-z]+)(?:\([^)]+\))?(?P<bang>!)?: \S")

def next_version(current, messages):
    major, minor, patch = (int(p) for p in current.split("."))
    kinds = [HEADER.match(m) for m in messages]
    if any(k and (k.group("bang") or "BREAKING CHANGE:" in m) for k, m in zip(kinds, messages)):
        return f"{major + 1}.0.0"
    if any(k and k.group("type") == "feat" for k in kinds):
        return f"{major}.{minor + 1}.0"
    if any(k and k.group("type") in ("fix", "perf") for k in kinds):
        return f"{major}.{minor}.{patch + 1}"
    return current

def sprint_report(name, planned, done_on, days, history, backlog_points, version, messages, date):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("PLANNED = {\"S3\": 3, \"S5\": 5, \"S6\": 2, \"S8\": 1}\nDONE = {\"S3\": 2, \"S6\": 3, \"S8\": 5}\nMSGS = [\"feat(storage): saved routes\", \"feat: import a map from CSV\", \"fix: count the header in line numbers\", \"test: fuzz the map checker\"]\n\ndef test_report():\n    \"the sprint in numbers\"\n    r = sprint_report(\"Sprint 1\", PLANNED, DONE, 5, [8], 20, \"0.1.0\", MSGS, \"2026-11-20\")\n    assert r[\"velocity\"] == 6 and r[\"history\"] == [8, 6] and r[\"carried\"] == [\"S5\"] and r[\"remaining\"] == [11, 11, 8, 6, 6, 5]\n    assert r[\"forecast\"] == {\"expected\": 4, \"best\": 4, \"worst\": 5} and r[\"release\"] == \"0.2.0\"\n    assert r[\"text\"] == \"\"\"# Sprint 1 review\nDone: S3, S6, S8 (6 points)\nCarried over: S5 (5 points)\nBurndown: 11 11 8 6 6 5\nVelocity: 6 (average of last 3: 7.0)\nForecast for 25 points: 4 sprints (best 4, worst 5)\nRelease: 0.2.0\n\"\"\"\n\ndef test_quiet_sprint():\n    \"nothing finished, nothing released\"\n    r = sprint_report(\"Sprint 2\", {\"S4\": 8}, {}, 5, [8, 6], 10, \"0.2.0\", [\"docs: notes\"], \"2026-11-27\")\n    assert \"Done: none (0 points)\" in r[\"text\"] and \"Carried over: S4 (8 points)\" in r[\"text\"] and r[\"release\"] is None\n    assert r[\"text\"].endswith(\"Release: nothing to release\\n\") and r[\"history\"] == [8, 6, 0]\n    assert \"Velocity: 0 (average of last 3: 4.7)\" in r[\"text\"] and \"Forecast for 18 points: 4 sprints (best 3, worst unknown)\" in r[\"text\"]")

> **Reading Waymark's first sprint.** Six points done out of eleven, and the burndown was flat for the first day and the last two: S5, the walk log, was the biggest story and never finished, because it was started on Monday alongside S3 rather than after it. The forecast now says four sprints for 25 points, with two build sprints left. That's the most useful sentence in the review: something must be cut **now**. For Waymark, S10 (exporting routes) moves to "won't have this time", and next sprint finishes S5 before starting anything else. What does your forecast say, and what will you cut?

### Exercise: Bonus: a demo that checks itself

`demo.txt` (a page file) is a demo script: each step is a line starting with `$ ` (the command), then the expected output lines, then optionally `[exit N]` (otherwise the exit code must be 0). Write `parse_demo(text)`: a list of `(command, expected_output, expected_code)`, where `expected_output` is the output lines joined with newlines and ending with one newline (or `""` if none). Then `run_demo(steps, main)`: run each step with the page's `cli.main(argv, out)` style function (`argv` is the command split on spaces, without the leading `waymark`) and return a list of report lines: `PASS N: COMMAND`, or `FAIL N: COMMAND` followed by `: expected exit E, got G` if the code differs, else `: output differs`. And `demo_ok(report)`: whether every line passes.

In [ ]:
import io

def parse_demo(text):
    ___

def run_demo(steps, main):
    ___

def demo_ok(report):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import io\nfrom cli import main\n\ndef test_parse():\n    \"steps from text\"\n    steps = parse_demo(open(\"demo.txt\").read())\n    assert len(steps) == 4\n    assert steps[0] == (\"waymark plan Mill Ridge\", \"Mill -> Weir -> Farm -> Ridge: 6.0 km, 1 h 39 min\\n\", 0)\n    assert steps[2] == (\"waymark plan Mill Moon\", \"unknown place: Moon\\n\", 2)\n    assert parse_demo(\"$ waymark plan\\n[exit 2]\\n\") == [(\"waymark plan\", \"\", 2)]\n\ndef test_run():\n    \"the demo against the real program\"\n    report = run_demo(parse_demo(open(\"demo.txt\").read()), main)\n    assert report == [\"PASS 1: waymark plan Mill Ridge\", \"PASS 2: waymark plan Ridge Mill\",\n                      \"PASS 3: waymark plan Mill Moon\", \"PASS 4: waymark plan Mill Island\"] and demo_ok(report)\n    stale = open(\"demo.txt\").read().replace(\"6.0 km, 1 h 39 min\", \"6.0 km, 1 h 30 min\").replace(\"[exit 1]\", \"[exit 3]\")\n    report = run_demo(parse_demo(stale), main)\n    assert report[0] == \"FAIL 1: waymark plan Mill Ridge: output differs\"\n    assert report[3] == \"FAIL 4: waymark plan Mill Island: expected exit 3, got 1\" and not demo_ok(report)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

## Mixed review

## The review

A sprint review has three parts, about an hour in all (shorter alone, but don't skip it):

1. **The demo.** Show each finished story **working**, from the user's side: the command, the request, the screen. Only finished stories (Week 87's definition of done) appear; "nearly done" is carried over, not demonstrated. The safest demo is one you've **run as a test**: a script of commands and their expected output, checked automatically, so the demo can't drift from the program. That's the bonus.
2. **The numbers.** Points finished (this sprint's **velocity**), what's carried over and why, the burndown's shape (a late cliff means stories were too big to finish one at a time), and the forecast for the rest of the backlog as a **range** from your slowest to your fastest sprint. With two sprints left in the capstone's build phase, the forecast tells you now whether the musts will fit, while there's still time to cut.
3. **The retrospective** (Week 87): what went well, what didn't, one to three actions for the next sprint, and whether last sprint's actions happened.

Then **release** what's done: a version from the commits (Week 87, Day 4), a changelog entry, and a tag. A release at the end of every sprint keeps the project always shippable.

## Bonus: a demo that checks itself

A demo that's typed live goes wrong in front of people; one that's described ("and then it would show...") proves nothing. The fix is a **demo script**: the commands you'll show and exactly what each prints, kept in the repository and **run as a test** in CI, like a golden file for the whole user experience. If the program changes, the test fails, and the demo is updated deliberately. It doubles as documentation: the README's usage section can be the same file. (Python's `doctest` does the same for examples in docstrings.)

## Weekly quiz

## Week 88 review

Tick off each one honestly. If one feels shaky, go back to that day.

- I can change a live schema with safe, numbered migrations, and build a repository, service and API routes for a new resource. ([Day 1](w088d1.html))
- I can validate records with dates properly, summarise them by month, personalise an estimate and count streaks. ([Day 2](w088d2.html))
- I can read CSV carefully, report errors and warnings by line, find disconnected parts and import in one transaction. ([Day 3](w088d3.html))
- I can use builders, oracles, golden files and fuzzing, and keep tests independent. ([Day 4](w088d4.html))
- I can review code in a sensible order, write labelled comments, read diffs and automate the mechanical checks. ([Day 5](w088d5.html))
- I can report a sprint's numbers, forecast the backlog, release, and keep a demo honest. (Today)

## Your capstone today

1. **The demo**: write a demo script for your finished stories and run it as a test in CI.
2. **The numbers**: velocity, carry-over, the burndown and a forecast for your remaining musts. If the worst case doesn't fit in the two sprints left, cut now and record it in the spec's out-of-scope list.
3. **Release**: tag a version with its changelog entry.
4. **Retrospective** and the plan for sprint 2 (Week 87's `plan_sprint`), with carried-over stories first.

Tomorrow is a **rest day**. Next week is **Capstone 3: the second build sprint**, with faster route search, round walks and walking preferences for Waymark.